# Υπολογιστική Όραση

Σκοπός της εργασίας είναι η υλοποίηση μιας εφαρμογής σε Python/OpenCV η οποία λαμβάνει εικόνα από κάμερα σε πραγματικό χρόνο και εφαρμόζει μεθόδους ανίχνευσης ακμών.

Οι μέθοδοι που θα χρησιμοποιηθούν είναι:

- Sobel
- Canny
- Laplacian

Μετά την ανίχνευση ακμών, η εφαρμογή θα βρίσκει τα περιγράμματα των αντικειμένων, δηλαδή τα contours.

Στη συνέχεια, με βάση απλά γεωμετρικά χαρακτηριστικά όπως το εμβαδόν, το bounding box και το aspect ratio, θα γίνεται απλή κατηγοριοποίηση αντικειμένων όπως:

- μπουκάλι
- τετράδιο
- κουτί

Η εφαρμογή δεν χρησιμοποιεί τεχνητή νοημοσύνη ή εκπαιδευμένο μοντέλο. Η αναγνώριση γίνεται με απλούς κανόνες, δηλαδή rule-based classification.

## Εισαγωγή βιβλιοθηκών

Σε αυτό το σημείο εισάγουμε τις απαραίτητες βιβλιοθήκες.

Το `cv2` είναι η OpenCV.

Το `numpy` χρησιμοποιείται για πράξεις πάνω σε πίνακες.

Το `time` χρησιμοποιείται για τον υπολογισμό των FPS.

## Εγκατάσταση βιβλιοθηκών

Για την υλοποίηση χρησιμοποιούμε τις βιβλιοθήκες:

- OpenCV
- NumPy

Η OpenCV χρησιμοποιείται για την επεξεργασία εικόνας, την κάμερα, τα φίλτρα ακμών και τα contours.

Η NumPy χρησιμοποιείται για αριθμητικές πράξεις πάνω στις εικόνες.

In [232]:
%pip install opencv-python numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [233]:
import cv2
import numpy as np
import time

## Βασικές ρυθμίσεις

Ορίζουμε κάποιες βασικές σταθερές για το πρόγραμμα.

Το `CAMERA_INDEX = 0` σημαίνει ότι χρησιμοποιούμε την πρώτη κάμερα του υπολογιστή.

Το `FRAME_WIDTH = 800` σημαίνει ότι κάθε frame θα γίνεται resize ώστε να έχει πλάτος 800 pixels.

Το `MIN_CONTOUR_AREA` χρησιμοποιείται για να αγνοούμε πολύ μικρά contours, επειδή συνήθως είναι θόρυβος.

Το `MAX_OBJECTS_TO_DRAW` περιορίζει πόσα αντικείμενα θα σχεδιάζονται στην εικόνα.

In [234]:
CAMERA_INDEX = 0
FRAME_WIDTH = 800

MIN_CONTOUR_AREA = 2500
MIN_BOX_AREA = 5000
MAX_OBJECTS_TO_DRAW = 6

SHOW_UNKNOWN_OBJECTS = True

LABEL_HOLD_SECONDS = 1.0
MATCH_DISTANCE = 90

## Resize του frame

Η κάμερα μπορεί να επιστρέφει εικόνες σε μεγάλο μέγεθος.

Για να τρέχει πιο γρήγορα η εφαρμογή, κάνουμε resize το frame σε σταθερό πλάτος.

Κρατάμε όμως το σωστό aspect ratio, ώστε να μη χαλάσει η εικόνα.

In [235]:
def resize_frame(frame, width=FRAME_WIDTH):
    """
    Κάνει resize το frame σε συγκεκριμένο πλάτος,
    κρατώντας σωστό aspect ratio.
    """
    
    height, original_width = frame.shape[:2]
    
    scale = width / original_width
    new_height = int(height * scale)
    
    resized = cv2.resize(frame, (width, new_height))
    
    return resized

## Προεπεξεργασία εικόνας

Πριν εφαρμόσουμε edge detection, κάνουμε δύο βασικά βήματα:

1. Μετατροπή της εικόνας σε grayscale.
2. Εφαρμογή Gaussian Blur.

Η μετατροπή σε grayscale γίνεται επειδή οι αλγόριθμοι ακμών δουλεύουν πάνω στις αλλαγές φωτεινότητας.

Το Gaussian Blur μειώνει τον θόρυβο, ώστε να μην ανιχνεύονται πολλές ψεύτικες ακμές.

In [236]:
def preprocess_frame(frame):
    """
    Μετατρέπει το frame σε grayscale, αυξάνει το contrast
    και εφαρμόζει blur.
    """

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Αύξηση τοπικού contrast
    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )
    enhanced = clahe.apply(gray)

    blurred = cv2.GaussianBlur(enhanced, (5, 5), 0)

    return blurred

## Κανονικοποίηση εικόνας

Οι μέθοδοι Sobel και Laplacian μπορεί να παράγουν τιμές που δεν είναι απευθείας κατάλληλες για εμφάνιση ως εικόνα.

Για αυτό, δημιουργούμε μια βοηθητική συνάρτηση που μετατρέπει το αποτέλεσμα σε εικόνα με τιμές από 0 έως 255.

In [237]:
def normalize_to_uint8(image):
    """
    Μετατρέπει μία εικόνα σε μορφή uint8 με τιμές από 0 έως 255.
    """
    
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    
    normalized = normalized.astype(np.uint8)
    
    return normalized

## Εφαρμογή Edge Detection

Σε αυτή τη συνάρτηση εφαρμόζουμε μία από τις τρεις μεθόδους:

- Canny
- Sobel
- Laplacian

Η επιλογή της μεθόδου γίνεται με τη μεταβλητή `method`.

In [238]:
def apply_edge_detection(gray_blurred, method):
    """
    Εφαρμόζει edge detection ανάλογα με τη μέθοδο που επιλέγουμε.
    
    method:
    - "canny"
    - "sobel"
    - "laplacian"
    """
    
    if method == "canny":
        edges = cv2.Canny(gray_blurred, 60, 160)
        return edges
    
    
    elif method == "sobel":
        # Sobel στον άξονα x
        grad_x = cv2.Sobel(gray_blurred, cv2.CV_64F, 1, 0, ksize=3)
        
        # Sobel στον άξονα y
        grad_y = cv2.Sobel(gray_blurred, cv2.CV_64F, 0, 1, ksize=3)
        
        # Συνδυασμός των δύο gradients
        magnitude = cv2.magnitude(grad_x, grad_y)
        
        # Μετατροπή σε εικόνα 0-255
        magnitude = normalize_to_uint8(magnitude)
        
        # Μετατροπή σε ασπρόμαυρη δυαδική εικόνα
        _, edges = cv2.threshold(
            magnitude,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        
        return edges
    
    
    elif method == "laplacian":
        # Εφαρμογή Laplacian
        lap = cv2.Laplacian(gray_blurred, cv2.CV_64F, ksize=3)
        
        # Παίρνουμε απόλυτες τιμές
        lap = np.absolute(lap)
        
        # Μετατροπή σε εικόνα 0-255
        lap = normalize_to_uint8(lap)
        
        # Μετατροπή σε δυαδική εικόνα
        _, edges = cv2.threshold(
            lap,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )
        
        return edges
    
    
    else:
        raise ValueError("Άγνωστη μέθοδος edge detection")

## Καθαρισμός ακμών

Μετά το edge detection, οι ακμές μπορεί να έχουν μικρά κενά.

Για να βγουν καλύτερα contours, χρησιμοποιούμε μορφολογικές πράξεις.

Το `MORPH_CLOSE` βοηθάει να κλείσουν μικρά κενά στις γραμμές.

Το `dilate` παχαίνει λίγο τις γραμμές, ώστε να ενώνονται καλύτερα τα περιγράμματα.

In [239]:
def clean_edges(edges):
    """
    Καθαρίζει και ενώνει τις ακμές ώστε να παράγονται πιο σταθερά contours.
    """

    close_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 9))
    small_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, close_kernel, iterations=2)

    dilated = cv2.dilate(closed, small_kernel, iterations=1)

    cleaned = cv2.morphologyEx(dilated, cv2.MORPH_OPEN, small_kernel, iterations=1)

    return cleaned

## Εξαγωγή χαρακτηριστικών από contour

Για κάθε contour υπολογίζουμε βασικά χαρακτηριστικά.

Τα πιο σημαντικά είναι:

- area: εμβαδόν
- perimeter: περίμετρος
- bounding box: ορθογώνιο που περικλείει το αντικείμενο
- width και height
- aspect ratio
- αριθμός κορυφών

Το aspect ratio υπολογίζεται ως:

height / width

Αυτό μας βοηθάει να ξεχωρίσουμε απλά σχήματα.

Για παράδειγμα:

- Ένα μπουκάλι είναι συνήθως ψηλό και στενό.
- Ένα τετράδιο είναι συνήθως πλατύ ορθογώνιο.
- Ένα κουτί είναι πιο συμπαγές ορθογώνιο ή τετράγωνο.

In [240]:
def extract_features(contour):
    """
    Υπολογίζει γεωμετρικά χαρακτηριστικά από ένα contour.
    """

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)

    x, y, w, h = cv2.boundingRect(contour)
    box_area = w * h

    aspect_ratio = h / float(w) if w != 0 else 0          # κάθετο ratio
    width_height_ratio = w / float(h) if h != 0 else 0    # οριζόντιο ratio

    extent = area / float(box_area) if box_area != 0 else 0

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / float(hull_area) if hull_area != 0 else 0

    rotated_rect = cv2.minAreaRect(contour)
    (_, _), (rw, rh), angle = rotated_rect

    long_side = max(rw, rh)
    short_side = min(rw, rh)

    long_aspect_ratio = long_side / short_side if short_side > 0 else 0
    rotated_box_area = rw * rh
    rotated_extent = area / rotated_box_area if rotated_box_area > 0 else 0

    circularity = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0

    epsilon = 0.03 * perimeter
    approx = cv2.approxPolyDP(contour, epsilon, True)
    vertices = len(approx)

    # ---------------------------------------------------------
    # Profile width features
    # Χρήσιμο για bottle vs phone.
    # Γεμίζουμε το contour σε μάσκα και μετράμε πλάτος πάνω/μέση/κάτω.
    # ---------------------------------------------------------
    mask = np.zeros((h, w), dtype=np.uint8)
    shifted_contour = contour - [x, y]
    cv2.drawContours(mask, [shifted_contour], -1, 255, -1)

    def region_width(y_start, y_end):
        region = mask[y_start:y_end, :]
        points = cv2.findNonZero(region)

        if points is None:
            return 0

        xs = points[:, 0, 0]
        return int(xs.max() - xs.min() + 1)

    top_width = region_width(0, max(1, h // 3))
    middle_width = region_width(h // 3, max(h // 3 + 1, 2 * h // 3))
    bottom_width = region_width(2 * h // 3, h)

    if middle_width > 0:
        top_middle_ratio = top_width / float(middle_width)
        bottom_middle_ratio = bottom_width / float(middle_width)
    else:
        top_middle_ratio = 0
        bottom_middle_ratio = 0

    features = {
        "area": area,
        "perimeter": perimeter,
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "box_area": box_area,
        "vertices": vertices,
        "aspect_ratio": aspect_ratio,
        "width_height_ratio": width_height_ratio,
        "extent": extent,
        "solidity": solidity,
        "long_aspect_ratio": long_aspect_ratio,
        "rotated_extent": rotated_extent,
        "circularity": circularity,
        "top_width": top_width,
        "middle_width": middle_width,
        "bottom_width": bottom_width,
        "top_middle_ratio": top_middle_ratio,
        "bottom_middle_ratio": bottom_middle_ratio
    }

    return features

In [241]:
def add_region_features(features, edges):
    """
    Προσθέτει χαρακτηριστικά από τις ακμές μέσα στο bounding box.
    Χρήσιμο για τετράδιο με σπιράλ, επειδή έχει πολλές ακμές αριστερά.
    """

    x = features["x"]
    y = features["y"]
    w = features["w"]
    h = features["h"]

    roi = edges[y:y+h, x:x+w]

    if roi.size == 0:
        features["left_edge_density"] = 0
        features["overall_edge_density"] = 0
        return features

    left_width = max(1, int(w * 0.25))
    left_strip = roi[:, :left_width]

    left_edge_density = cv2.countNonZero(left_strip) / float(left_strip.size)
    overall_edge_density = cv2.countNonZero(roi) / float(roi.size)

    features["left_edge_density"] = left_edge_density
    features["overall_edge_density"] = overall_edge_density

    return features

## Απλή αναγνώριση αντικειμένων

Σε αυτό το σημείο δεν χρησιμοποιούμε machine learning.

Χρησιμοποιούμε απλούς κανόνες.

Η λογική είναι:

- Αν το αντικείμενο είναι ψηλό και στενό, το θεωρούμε πιθανό μπουκάλι.
- Αν έχει περίπου 4 κορυφές και είναι πλατύ, το θεωρούμε πιθανό τετράδιο.
- Αν έχει περίπου 4 κορυφές και είναι πιο συμπαγές, το θεωρούμε πιθανό κουτί.

Αυτό ονομάζεται rule-based classification.

Είναι σημαντικό να αναφέρουμε στην εργασία ότι αυτή η μέθοδος λειτουργεί σε ελεγχόμενο περιβάλλον και όχι σε οποιαδήποτε εικόνα.

In [242]:
def classify_object(features):
    """
    Πιο σταθερή rule-based κατηγοριοποίηση.
    Δίνουμε προτεραιότητα στο Notebook όταν το αντικείμενο είναι μεγάλο,
    γεμάτο και επίπεδο ορθογώνιο/τετράγωνο στην εικόνα.
    """

    box_area = features["box_area"]
    vertices = features["vertices"]

    extent = features["extent"]
    solidity = features["solidity"]
    long_ar = features["long_aspect_ratio"]
    rotated_extent = features["rotated_extent"]

    w = features["w"]
    h = features["h"]

    width_height_ratio = w / float(h) if h != 0 else 0
    height_width_ratio = h / float(w) if w != 0 else 0

    top_middle_ratio = features.get("top_middle_ratio", 1.0)
    bottom_middle_ratio = features.get("bottom_middle_ratio", 1.0)

    # ---------------------------------------------------
    # 1. Sunglasses
    # Πολύ φαρδύ και χαμηλό αντικείμενο.
    # ---------------------------------------------------
    if (
        long_ar >= 2.2
        and height_width_ratio <= 0.65
        and 4000 < box_area < 60000
        and 0.10 < extent < 0.85
        and solidity > 0.25
        and 4 <= vertices <= 20
    ):
        label = "Sunglasses"
        color = (255, 0, 255)

    # ---------------------------------------------------
    # 2. Bottle
    # Ψηλό/στενό αντικείμενο, όχι τέλειο ορθογώνιο.
    # ---------------------------------------------------
    elif (
        height_width_ratio >= 1.55
        and long_ar >= 1.6
        and box_area > 8000
        and extent > 0.15
        and solidity > 0.35
        and rotated_extent < 0.82
        and (
            top_middle_ratio < 0.88
            or bottom_middle_ratio < 0.92
        )
    ):
        label = "Bottle"
        color = (0, 255, 0)

    # ---------------------------------------------------
    # 3. Notebook
    # Μεγάλο, γεμάτο, επίπεδο αντικείμενο.
    # Το βάζουμε ΠΡΙΝ από το Box.
    # Ακόμα και αν φαίνεται σχεδόν τετράγωνο στην κάμερα,
    # αν είναι μεγάλο και γεμάτο, το θεωρούμε Notebook.
    # ---------------------------------------------------
    elif (
        box_area > 50000
        and 1.0 <= long_ar <= 3.2
        and extent > 0.55
        and solidity > 0.70
        and rotated_extent > 0.55
        and 4 <= vertices <= 14
    ):
        label = "Notebook"
        color = (255, 0, 0)

    # ---------------------------------------------------
    # 4. Rectangular Object
    # Μικρότερο ορθογώνιο αντικείμενο, π.χ. κινητό.
    # ---------------------------------------------------
    elif (
        1.45 <= long_ar <= 3.2
        and 12000 < box_area <= 50000
        and extent > 0.45
        and solidity > 0.60
        and rotated_extent > 0.45
        and 4 <= vertices <= 14
    ):
        label = "Rectangular Object"
        color = (255, 0, 0)

    # ---------------------------------------------------
    # 5. Box
    # Το κάνουμε πολύ πιο αυστηρό.
    # Το Box θα πιάνει κυρίως μικρότερα, συμπαγή, σχεδόν τετράγωνα αντικείμενα.
    # ---------------------------------------------------
    elif (
        0.80 <= width_height_ratio <= 1.25
        and 0.80 <= height_width_ratio <= 1.25
        and 10000 < box_area <= 50000
        and extent > 0.45
        and solidity > 0.60
        and 4 <= vertices <= 14
    ):
        label = "Box"
        color = (0, 165, 255)

    else:
        label = "Unknown"
        color = (255, 255, 255)

    return label, color

## Εύρεση και φιλτράρισμα contours

Η OpenCV βρίσκει όλα τα contours στην εικόνα.

Όμως πολλά από αυτά μπορεί να είναι θόρυβος.

Για αυτό κάνουμε φιλτράρισμα:

- Αγνοούμε contours με πολύ μικρό εμβαδόν.
- Αγνοούμε πολύ μικρά bounding boxes.
- Κρατάμε μόνο τα μεγαλύτερα contours.

In [243]:
def find_valid_contours(edges, frame_shape):
    """
    Βρίσκει contours και κρατάει μόνο όσα μοιάζουν με πραγματικά αντικείμενα.
    """

    contours, _ = cv2.findContours(
        edges,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    frame_height, frame_width = frame_shape[:2]
    frame_area = frame_height * frame_width

    valid_contours = []

    for contour in contours:
        features = extract_features(contour)

        area = features["area"]
        x = features["x"]
        y = features["y"]
        w = features["w"]
        h = features["h"]
        box_area = features["box_area"]
        aspect_ratio = features["aspect_ratio"]
        extent = features["extent"]
        solidity = features["solidity"]

        # Πετάμε μικρά contours
        if area < MIN_CONTOUR_AREA:
            continue

        # Πετάμε μικρά bounding boxes
        if box_area < MIN_BOX_AREA:
            continue

        # Πετάμε υπερβολικά μεγάλα contours, π.χ. τραπέζι/φόντο
        if box_area > 0.45 * frame_area:
            continue

        # Πετάμε πολύ μικρές διαστάσεις
        if w < 45 or h < 45:
            continue

        # Πετάμε contours που ακουμπάνε στα όρια της εικόνας
        margin = 8
        if x <= margin or y <= margin:
            continue

        if x + w >= frame_width - margin or y + h >= frame_height - margin:
            continue

        # Πετάμε υπερβολικά λεπτές γραμμές
        if aspect_ratio < 0.15 or aspect_ratio > 6.0:
            continue

        # Πετάμε πολύ άδεια/σπασμένα contours
        if extent < 0.18:
            continue

        # Πετάμε πολύ ακανόνιστα contours
        if solidity < 0.45:
            continue

        valid_contours.append(contour)

    valid_contours = sorted(
        valid_contours,
        key=lambda c: cv2.boundingRect(c)[2] * cv2.boundingRect(c)[3],
        reverse=True
    )

    return valid_contours[:MAX_OBJECTS_TO_DRAW]

## Σχεδίαση αντικειμένου πάνω στο frame

Για κάθε αντικείμενο σχεδιάζουμε:

- το contour
- το bounding box
- την ετικέτα του αντικειμένου
- το εμβαδόν
- τον αριθμό κορυφών

In [244]:
def draw_object_info(frame, contour, features, label, color):
    """
    Σχεδιάζει το contour, το bounding box και πληροφορίες ταξινόμησης.
    """

    x = features["x"]
    y = features["y"]
    w = features["w"]
    h = features["h"]

    orientation = "Vertical" if h > w else "Horizontal"

    text1 = f"{label}"
    text2 = f"{orientation} | LAR:{features['long_aspect_ratio']:.2f}"
    text3 = f"E:{features['extent']:.2f} S:{features['solidity']:.2f}"

    cv2.drawContours(frame, [contour], -1, color, 2)
    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

    text1 = f"{label}"
    text2 = f"AR:{features['aspect_ratio']:.2f} LAR:{features['long_aspect_ratio']:.2f}"
    text3 = f"E:{features['extent']:.2f} S:{features['solidity']:.2f} V:{features['vertices']}"

    cv2.putText(
        frame,
        text1,
        (x, max(y - 35, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        color,
        2
    )

    cv2.putText(
        frame,
        text2,
        (x, max(y - 15, 40)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        color,
        1
    )

    cv2.putText(
        frame,
        text3,
        (x, y + h + 18),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.45,
        color,
        1
    )

## Εμφάνιση πληροφοριών στο παράθυρο

Στο πάνω μέρος του frame εμφανίζουμε:

- τη μέθοδο edge detection
- τα FPS
- τον αριθμό των contours
- τα πλήκτρα χειρισμού

In [245]:
def draw_header(frame, method, fps, contour_count):
    """
    Εμφανίζει πληροφορίες στο πάνω μέρος του frame.
    """
    
    text1 = f"Method: {method.upper()} | FPS: {fps:.1f} | Contours: {contour_count}"
    text2 = "Keys: 1=Canny, 2=Sobel, 3=Laplacian, q/ESC=Exit"
    
    cv2.putText(
        frame,
        text1,
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )
    
    cv2.putText(
        frame,
        text2,
        (15, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (0, 255, 255),
        2
    )

In [246]:
STOP_BUTTON_BOX = None
app_state = {
    "stop": False
}


def draw_stop_button(frame):
    """
    Σχεδιάζει κουμπί τερματισμού πάνω δεξιά στο frame.
    """

    global STOP_BUTTON_BOX

    button_width = 160
    button_height = 45
    margin = 15

    x1 = frame.shape[1] - button_width - margin
    y1 = margin
    x2 = x1 + button_width
    y2 = y1 + button_height

    STOP_BUTTON_BOX = (x1, y1, x2, y2)

    # Κόκκινο γεμάτο κουμπί
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), -1)

    # Άσπρο περίγραμμα
    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)

    # Κείμενο κουμπιού
    cv2.putText(
        frame,
        "TERMINATE",
        (x1 + 15, y1 + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )


def mouse_callback(event, x, y, flags, param):
    """
    Ελέγχει αν ο χρήστης έκανε click πάνω στο κουμπί TERMINATE.
    """

    global STOP_BUTTON_BOX

    if event == cv2.EVENT_LBUTTONDOWN and STOP_BUTTON_BOX is not None:
        x1, y1, x2, y2 = STOP_BUTTON_BOX

        if x1 <= x <= x2 and y1 <= y <= y2:
            app_state["stop"] = True
            print("Πατήθηκε το κουμπί TERMINATE.")

## Κύρια εφαρμογή

Αυτή είναι η βασική συνάρτηση του προγράμματος.

Η ροή της εφαρμογής είναι:

1. Άνοιγμα κάμερας.
2. Λήψη frame.
3. Resize.
4. Grayscale και blur.
5. Edge detection.
6. Καθαρισμός ακμών.
7. Εύρεση contours.
8. Εξαγωγή χαρακτηριστικών.
9. Κατηγοριοποίηση αντικειμένων.
10. Σχεδίαση αποτελεσμάτων.
11. Υπολογισμός FPS.

Με τα πλήκτρα:

- 1 επιλέγουμε Canny.
- 2 επιλέγουμε Sobel.
- 3 επιλέγουμε Laplacian.
- q ή ESC κλείνουμε το πρόγραμμα.

In [247]:
def get_box_center(features):
    """
    Υπολογίζει το κέντρο του bounding box.
    """
    cx = features["x"] + features["w"] // 2
    cy = features["y"] + features["h"] // 2
    return cx, cy


def find_previous_label(features, label_memory, current_time):
    """
    Αν το τρέχον contour βγει Unknown, ψάχνει αν κοντά του
    υπήρχε πρόσφατα γνωστό label.
    """

    cx, cy = get_box_center(features)

    best_match = None
    best_distance = float("inf")

    for item in label_memory:
        old_cx, old_cy = item["center"]
        age = current_time - item["time"]

        if age > LABEL_HOLD_SECONDS:
            continue

        distance = ((cx - old_cx) ** 2 + (cy - old_cy) ** 2) ** 0.5

        if distance < MATCH_DISTANCE and distance < best_distance:
            best_distance = distance
            best_match = item

    return best_match


def update_label_memory(features, label, color, label_memory, current_time):
    """
    Αποθηκεύει πρόσφατα αναγνωρισμένα αντικείμενα.
    """

    if label == "Unknown":
        return

    center = get_box_center(features)

    label_memory.append({
        "label": label,
        "color": color,
        "center": center,
        "time": current_time
    })

    # Κρατάμε μόνο πρόσφατες αναγνωρίσεις
    label_memory[:] = [
        item for item in label_memory
        if current_time - item["time"] <= LABEL_HOLD_SECONDS
    ]

In [248]:
def run_realtime_app():
    """
    Εκτελεί την εφαρμογή real-time video.
    Η εφαρμογή κλείνει είτε με:
    - click στο κουμπί TERMINATE
    - q
    - ESC
    - κλείσιμο του παραθύρου
    """

    app_state["stop"] = False

    window_name = "Real-Time Object Contour Detection"
    edges_window_name = "Edges"

    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("Δεν μπόρεσε να ανοίξει η κάμερα.")
        return

    label_memory = []
    method = "canny"
    previous_time = time.time()

    print("Η εφαρμογή ξεκίνησε.")
    print("Πάτησε 1 για Canny.")
    print("Πάτησε 2 για Sobel.")
    print("Πάτησε 3 για Laplacian.")
    print("Πάτησε το κουμπί TERMINATE ή q για έξοδο.")

    try:
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        cv2.namedWindow(edges_window_name, cv2.WINDOW_NORMAL)

        # Ενεργοποίηση mouse click στο βασικό παράθυρο
        cv2.setMouseCallback(window_name, mouse_callback)

        while not app_state["stop"]:
            ret, frame = cap.read()

            if not ret:
                print("Δεν μπόρεσε να διαβαστεί frame από την κάμερα.")
                break

            # Resize frame
            frame = resize_frame(frame)

            # Αντιγραφή για να σχεδιάσουμε πάνω σε αυτή
            output = frame.copy()

            # Προεπεξεργασία
            gray_blurred = preprocess_frame(frame)

            # Edge detection
            edges = apply_edge_detection(gray_blurred, method)

            # Καθαρισμός ακμών
            cleaned_edges = clean_edges(edges)

            # Εύρεση contours
            contours = find_valid_contours(cleaned_edges, frame.shape)

            # Για κάθε contour βρίσκουμε χαρακτηριστικά και το ταξινομούμε
            current_time = time.time()

            for contour in contours:
                features = extract_features(contour)

                label, color = classify_object(features)

                # Αν αναγνωρίστηκε σωστά, το αποθηκεύουμε προσωρινά
                if label != "Unknown":
                    update_label_memory(features, label, color, label_memory, current_time)

                # Αν τώρα βγήκε Unknown, αλλά πριν λίγο ήταν γνωστό αντικείμενο,
                # κρατάμε το προηγούμενο label για λίγο
                elif label == "Unknown":
                    previous_match = find_previous_label(features, label_memory, current_time)

                    if previous_match is not None:
                        label = previous_match["label"]
                        color = previous_match["color"]

                if label == "Unknown" and not SHOW_UNKNOWN_OBJECTS:
                    continue

                draw_object_info(output, contour, features, label, color)

            # Υπολογισμός FPS
            current_time = time.time()
            elapsed_time = current_time - previous_time

            if elapsed_time > 0:
                fps = 1 / elapsed_time
            else:
                fps = 0

            previous_time = current_time

            # Εμφάνιση πληροφοριών
            draw_header(output, method, fps, len(contours))

            # Σχεδίαση κουμπιού τερματισμού
            draw_stop_button(output)

            # Παράθυρα OpenCV
            cv2.imshow(window_name, output)
            cv2.imshow(edges_window_name, cleaned_edges)

            # Έλεγχος πλήκτρων
            key = cv2.waitKey(10) & 0xFF

            if key == ord("1"):
                method = "canny"
                print("Μέθοδος: Canny")

            elif key == ord("2"):
                method = "sobel"
                print("Μέθοδος: Sobel")

            elif key == ord("3"):
                method = "laplacian"
                print("Μέθοδος: Laplacian")

            elif key == ord("q") or key == 27:
                app_state["stop"] = True
                print("Ζητήθηκε έξοδος από το πληκτρολόγιο.")

            # Αν κλείσεις το παράθυρο με Χ, να σταματάει και το loop
            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                app_state["stop"] = True
                print("Το παράθυρο έκλεισε.")

    finally:
        cap.release()
        cv2.destroyAllWindows()

        # Βοηθάει σε Mac/Jupyter να κλείσουν σωστά τα OpenCV windows
        for _ in range(5):
            cv2.waitKey(1)

        print("Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.")

## Εκτέλεση εφαρμογής

Τρέχουμε το παρακάτω cell για να ξεκινήσει η εφαρμογή.

Η κάμερα θα ανοίξει σε ξεχωριστό παράθυρο OpenCV.

Για να κλείσει η εφαρμογή, πατάμε `q` ή `ESC` πάνω στο παράθυρο της OpenCV.

In [249]:
run_realtime_app()

Η εφαρμογή ξεκίνησε.
Πάτησε 1 για Canny.
Πάτησε 2 για Sobel.
Πάτησε 3 για Laplacian.
Πάτησε το κουμπί TERMINATE ή q για έξοδο.
Πατήθηκε το κουμπί TERMINATE.
Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.


## Πειραματική σύγκριση

Κατά τη διάρκεια των δοκιμών, συγκρίνουμε τις τρεις μεθόδους:

| Μέθοδος | Ποιότητα ακμών | Καθαρότητα contours | FPS | Παρατηρήσεις |
|---|---|---|---|---|
| Canny | | | | |
| Sobel | | | | |
| Laplacian | | | | |

Σκοπός δεν είναι μόνο να δείξουμε ότι λειτουργεί ο κώδικας, αλλά να συγκρίνουμε ποια μέθοδος είναι καλύτερη για την εξαγωγή contours σε πραγματικό χρόνο.

## Συμπέρασμα

Η εφαρμογή υλοποιεί μια απλή μέθοδο υπολογιστικής όρασης για real-time ανίχνευση ακμών και εξαγωγή περιγραμμάτων αντικειμένων.

Μετά την εφαρμογή των φίλτρων Sobel, Canny και Laplacian, βρίσκονται contours και υπολογίζονται γεωμετρικά χαρακτηριστικά.

Με βάση αυτά τα χαρακτηριστικά εφαρμόζεται απλή rule-based κατηγοριοποίηση αντικειμένων.

Η μέθοδος αυτή δεν αποτελεί γενικό σύστημα αναγνώρισης αντικειμένων, αλλά λειτουργεί σε ελεγχόμενο περιβάλλον και δείχνει πώς μπορούν να χρησιμοποιηθούν οι κλασικές τεχνικές υπολογιστικής όρασης για την ανίχνευση και βασική ταξινόμηση αντικειμένων.